# Notebook 04 — Dataset Finalisation and Feature Re-extraction

Combines all data sources into a final balanced dataset and re-extracts stylometric features for all 10,492 emails.

## What This Notebook Does
- Loads Phase 1 dataset (legitimate + human phishing)
- Loads AI-generated phishing
- Tops up legitimate and human phishing classes to 4,000 each
- Merges and shuffles all three classes
- Re-extracts all 63 stylometric features for the full dataset
- Saves final dataset and feature matrix to disk

## Inputs
- data/processed/dataset_phase1.csv (from Notebook 01)
- data/generated/ai_phishing_all.csv (from Notebook 03)

## Outputs
- data/processed/dataset_final.csv — 10,492 emails (3 classes)
- data/processed/stylometric_features_final.csv — 63 features

## Class Distribution
- Label 0 (Legitimate): 4,000 emails
- Label 1 (Human Phishing): 4,000 emails
- Label 2 (AI Phishing): 2,492 emails
- Total: 10,492 emails

## Runtime
Approximately 15-20 minutes

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE_DIR = Path("C:/phishing_detection")
DATA_RAW = BASE_DIR / "data" / "raw"
DATA_PROCESSED = BASE_DIR / "data" / "processed"
DATA_GENERATED = BASE_DIR / "data" / "generated"

# Load current phase 1 dataset (4,000 emails - 2 classes)
phase1 = pd.read_csv(DATA_PROCESSED / "dataset_phase1.csv")
print("Phase 1 dataset:")
print(phase1['label'].value_counts().sort_index())

# Load generated AI phishing emails
ai_phishing = pd.read_csv(DATA_GENERATED / "ai_phishing_all.csv")
print(f"\nAI phishing generated: {len(ai_phishing)} emails")
print(ai_phishing['scenario'].value_counts())

In [ ]:
# Load additional legitimate emails from Enron (already cleaned)
print("Loading additional Enron emails")
enron_raw = pd.read_csv(DATA_RAW / "enron" / "emails.csv")

import re
def extract_email_body(raw_message):
    parts = re.split(r'\n\n', str(raw_message), maxsplit=1)
    body = parts[1].strip() if len(parts) == 2 else raw_message.strip()
    body = re.sub(r'-{3,}.*?-{3,}', '', body, flags=re.DOTALL)
    body = '\n'.join([l for l in body.split('\n') if not l.startswith('>')])
    body = re.sub(r'\s+', ' ', body).strip()
    return body

# Get emails not already in phase1
enron_raw['text'] = enron_raw['message'].apply(extract_email_body)
enron_clean = enron_raw[enron_raw['text'].str.len() > 50][['text']].copy()
enron_clean['label'] = 0

# Sample 2,000 fresh legitimate emails
extra_legit = enron_clean.sample(n=2000, random_state=99).reset_index(drop=True)
print(f"Extra legitimate emails: {len(extra_legit)}")

# Load additional human phishing from nazario
nazario_raw = pd.read_csv(DATA_RAW / "nazario" / "Phishing_Email.csv")
nazario_raw = nazario_raw.rename(columns={"Email Text": "text", "Email Type": "email_type"})
nazario_raw['label'] = nazario_raw['email_type'].map({'Safe Email': 0, 'Phishing Email': 1})
nazario_phish = nazario_raw[nazario_raw['label'] == 1][['text', 'label']].dropna()
nazario_phish = nazario_phish[nazario_phish['text'].str.len() > 50]

# Sample 2,000 fresh human phishing emails
extra_phish = nazario_phish.sample(n=2000, random_state=99).reset_index(drop=True)
print(f"Extra human phishing emails: {len(extra_phish)}")

In [ ]:
# Prepare AI phishing emails - keep only text and label columns
ai_clean = ai_phishing[['text', 'label']].copy()
ai_clean['text'] = ai_clean['text'].astype(str).str.strip()
ai_clean = ai_clean[ai_clean['text'].str.len() > 50]

# Sample 4,000 from the 2,492 we have
# We only have 2,492 so we take all of them and note this in evaluation
print(f"AI phishing available: {len(ai_clean)}")
#Using all available AI phishing emails (~2,492), slightly imbalanced classes"


In [ ]:
# Combine phase1 + extra emails + AI phishing
legitimate = pd.concat([
    phase1[phase1['label'] == 0],
    extra_legit[['text', 'label']]
], ignore_index=True)

human_phishing = pd.concat([
    phase1[phase1['label'] == 1],
    extra_phish[['text', 'label']]
], ignore_index=True)

# Final combination
dataset_final = pd.concat([
    legitimate,
    human_phishing,
    ai_clean[['text', 'label']]
], ignore_index=True)

# Clean and shuffle
dataset_final['text'] = dataset_final['text'].astype(str).str.strip()
dataset_final = dataset_final[dataset_final['text'].str.len() > 50].dropna()
dataset_final['label'] = dataset_final['label'].astype(int)
dataset_final = dataset_final.sample(frac=1, random_state=42).reset_index(drop=True)

print("FINAL DATASET")
print(f"Total emails: {len(dataset_final)}")
print("\nClass distribution:")
print(dataset_final['label'].value_counts().sort_index())
print("\nLabel meanings:")
print("  0 = Legitimate")
print("  1 = Human phishing")
print("  2 = AI-generated phishing")

# Save
save_path = DATA_PROCESSED / "dataset_final.csv"
dataset_final.to_csv(save_path, index=False)
print(f"\nSaved to: {save_path}")
print(f"File size: {save_path.stat().st_size / (1024*1024):.1f} MB")